In [26]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.layers import Input, Embedding, GlobalAveragePooling1D, Dense, Dropout, Bidirectional, LSTM
from tensorflow.keras.models import Model


In [27]:
df1 = pd.read_csv('linkedin_jobs_20260418_215155.csv')
df2 = pd.read_csv('linkedin_jobs_20260416_223428.csv')
df3 = pd.read_csv('jobstreet_jobs_20260420_205621.csv')
df = pd.concat([df1, df2, df3])
rows, cols = df.shape

print(f"Rows : {rows}")
print(f"Cols : {cols}")
df.info()

Rows : 1895
Cols : 12
<class 'pandas.core.frame.DataFrame'>
Index: 1895 entries, 0 to 63
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                1895 non-null   int64 
 1   title             1895 non-null   object
 2   company           1894 non-null   object
 3   location          1831 non-null   object
 4   job_description   1895 non-null   object
 5   job_url           1895 non-null   object
 6   search_role       1895 non-null   object
 7   search_location   1895 non-null   object
 8   extracted_skills  1895 non-null   object
 9   skills_count      1895 non-null   int64 
 10  scraped_at        1895 non-null   object
 11  salary            29 non-null     object
dtypes: int64(2), object(10)
memory usage: 192.5+ KB


In [28]:
df = df.drop_duplicates()
df['company'] = df['company'].fillna(df['company'].mode()[0])
df['location'] = df['location'].fillna(df['location'].mode()[0])
df = df.drop(columns='salary', errors='ignore')

print(f"total duplicated columns : {df.duplicated().sum()}")
print(f"total null column : {df.isnull().sum()}")

total duplicated columns : 0
total null column : id                  0
title               0
company             0
location            0
job_description     0
job_url             0
search_role         0
search_location     0
extracted_skills    0
skills_count        0
scraped_at          0
dtype: int64


In [29]:
df["job_description"] = df["job_description"].fillna("").astype(str)
df["extracted_skills"] = df["extracted_skills"].fillna("").astype(str)

df["text_input"] = df["job_description"] + " " + df["extracted_skills"]

encoder = LabelEncoder()
df['label'] = encoder.fit_transform(df['search_role'])

num_classes = len(encoder.classes_)
print(f"Total role unik: {num_classes}")

print(f"Mapping kelas:\n{dict(zip(encoder.classes_, range(num_classes)))}")

X_train, X_val, y_train, y_val = train_test_split(
    df['text_input'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

max_vocab_length = 10000
max_sequence_length = 250

vectorizer = TextVectorization(
    max_tokens=max_vocab_length,
    output_mode='int',
    output_sequence_length=max_sequence_length
)

vectorizer.adapt(X_train.to_numpy())

print("\nShape X_train:", X_train.shape)
print("Contoh kalimat pertama setelah di-vektorisasi (angka):")
print(vectorizer(X_train.iloc[0:1]).numpy())

Total role unik: 3
Mapping kelas:
{'Data Scientist': 0, 'Web Developer': 1, 'web': 2}

Shape X_train: (1516,)
Contoh kalimat pertama setelah di-vektorisasi (angka):
[[  21    4   19   89    5 2464    4  426  116  294 4166 3737  115    5
  1662   61    2  294  768    8 5202 5279  931 3710  198 1093  503    2
   369  184  115 2220   32 4991    7  650    6 4762  295  971   13  419
   538   15   31  935  133    3  167    8  535   92    2  345  426   12
     3   14  295   17    8   88  960   16   25   46    4  122    3  951
     5 1094 2250  114   23  105   16    7    8 2684    6  205  787   12
    16   25  744  679   26   81 2381  124    5    8  430   22   69    2
    70    8 2701   65    6   28  185 1772   90   25   32 2366   20    8
  2300  699   13  745   16    3  158 1342  672  271  425   32  193    3
  1653    3   30  341  238  811 1769  366 1444   91    4 2140   27  172
    13  395   20 8851    2 1481 2690  746    1 1674 2138    3  329    1
   538 2454   13   22  220    7  706    2 2

In [30]:
print(type(X_train), X_train.dtype)
print(X_train.head(3))
print(X_train.astype(str).to_numpy().dtype)

<class 'pandas.core.series.Series'> object
578    About the job\n\nResponsibilities\n\nIn partic...
89     About the job\n\nJob Description\n\n1. Identif...
361    About the job\n\nJob Description\n\n\nDrive co...
Name: text_input, dtype: object
object


In [31]:
import numpy as np

class CustomEarlyStopping(tf.keras.callbacks.Callback):
    def __init__(self, patience=3):
        super(CustomEarlyStopping, self).__init__()
        self.patience = patience
        self.best_weights = None
        self.best_loss = np.inf
        self.wait = 0

    def on_epoch_end(self, epoch, logs=None):
        current_loss = logs.get('val_loss')

        if current_loss < self.best_loss:
            self.best_loss = current_loss
            self.wait = 0
            self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f"\n[INFO] val_loss tidak membaik selama {self.patience} epoch. Overfitting terdeteksi. Stop training!")
                self.model.stop_training = True
                print("[INFO] Mengembalikan bobot model ke epoch terbaik.")
                self.model.set_weights(self.best_weights)

In [32]:
vocab_size = len(vectorizer.get_vocabulary())
embedding_dim = 32

inputs = Input(shape=(1,), dtype=tf.string, name='text_input')
x = vectorizer(inputs)

x = Embedding(input_dim=vocab_size, output_dim=embedding_dim, name='embedding_layer')(x)

x = Bidirectional(LSTM(32, return_sequences=False))(x)

x = Dense(64, activation='relu')(x)
x = Dropout(0.3)(x)

outputs = Dense(num_classes, activation='softmax', name='role_output')(x)

model = Model(inputs=inputs, outputs=outputs, name="Job_Role_Classifier_LSTM")

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nMulai proses training...")

history = model.fit(
    X_train.to_numpy(), y_train,
    validation_data=(X_val.to_numpy(), y_val),
    epochs=15,
    batch_size=32,
    callbacks=[CustomEarlyStopping(patience=3)]
)


Mulai proses training...
Epoch 1/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 15s 222ms/step - accuracy: 0.5092 - loss: 0.9111 - val_accuracy: 0.5277 - val_loss: 0.7918
Epoch 2/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 8s 176ms/step - accuracy: 0.5686 - loss: 0.7495 - val_accuracy: 0.6306 - val_loss: 0.6735
Epoch 3/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 10s 208ms/step - accuracy: 0.8212 - loss: 0.4612 - val_accuracy: 0.8628 - val_loss: 0.3572
Epoch 4/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 10s 208ms/step - accuracy: 0.9156 - loss: 0.2444 - val_accuracy: 0.8338 - val_loss: 0.3799
Epoch 5/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 10s 206ms/step - accuracy: 0.9327 - loss: 0.1908 - val_accuracy: 0.8839 - val_loss: 0.3243
Epoch 6/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 9s 180ms/step - accuracy: 0.9485 - loss: 0.1430 - val_accuracy: 0.8813 - val_loss: 0.3956
Epoch 7/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 13s 225ms/step - accuracy: 0.9631 - loss: 0.1133 - val_accuracy: 0.8786 - val_loss: 0.3377
Epoch 8/15
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step - accuracy: 0.9657 - 

In [34]:
model_path = 'job_role_model.keras'
model.save(model_path)
print(f"[INFO] Model berhasil diekspor ke: {model_path}")

loaded_model = tf.keras.models.load_model(model_path)

def rekomendasi_role(teks_skill_user):
    input_tensor = tf.constant([teks_skill_user], dtype=tf.string)

    pred_probs = loaded_model.predict(input_tensor, verbose=0)

    pred_class_idx = np.argmax(pred_probs)

    predicted_role = encoder.inverse_transform([pred_class_idx])[0]

    return predicted_role

# 3. Test fungsi inference
skill_input = "saya punya pengalaman bikin antarmuka web pakai reactjs, next.js, tailwind, dan biasa pakai linux ubuntu"
hasil_prediksi = rekomendasi_role(skill_input)

print("\n--- Hasil Inference ---")
print(f"Skill User : {skill_input}")
print(f"Cocok untuk: {hasil_prediksi}")

[INFO] Model berhasil diekspor ke: job_role_model.keras

--- Hasil Inference ---
Skill User : saya punya pengalaman bikin antarmuka web pakai reactjs, next.js, tailwind, dan biasa pakai linux ubuntu
Cocok untuk: web
